In [81]:
import matplotlib.pyplot as plt
import sys
from typing import List, Dict
if not '../..' in sys.path:
    sys.path.append('../..')
import basico
import numpy
import pandas
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import random
import itertools
import math
import tqdm
import libsbml
import pickle
import os
import sklearn

In [82]:
INPUTS = 3
OUTPUTS = 11

class ChassFfnnRegressor(nn.Module):
    def __init__(self, inputs, layers, outputs, activation):
        super().__init__()
        all_layers = [inputs] + layers + [outputs]
        self.linears = nn.ModuleList([nn.Linear(all_layers[i], all_layers[i + 1]) for i in range(len(all_layers) - 1)])
        if activation == "tanh":
            self.act = nn.Tanh()
        elif activation == "relu":
            self.act = nn.ReLU()
    
    def forward(self, x):
        for hidden in self.linears[:-1]:
            x = self.act(hidden(x))
        x = self.linears[-1](x)
        return x

nums_nodes_layers_s_ffnn = [[5,5], [10,10], [16,16], [16,16,16]]
#nn_models = [ChassFfnnRegressor(INPUTS, layers, OUTPUTS) for layers in nums_nodes_layers_s]
loss_fn = nn.MSELoss()

In [83]:
NUM_HIDDEN = 1

class ChassRnnRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, activation):
        super(ChassRnnRegressor, self).__init__()
        
        # Number of hidden dimensions
        self.hidden_dim = hidden_dim
        
        # Number of hidden layers
        self.layer_dim = layer_dim
        
        # RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity=activation)
        
        # Readout layer
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim)
            
        # One time step
        out, hn = self.rnn(x, h0)
        out = self.fc(out) 
        return out


nums_nodes_one_layer_ernn = [10, 15, 18, 21]
#nn_models = [ChassRnnRegressor(INPUTS, num_nodes, NUM_HIDDEN, OUTPUTS) for num_nodes in nums_nodes_one_layer]

In [84]:
num_nodes = {"ffnn": nums_nodes_layers_s_ffnn,
             "ernn": nums_nodes_one_layer_ernn}

In [85]:
def load_model(path_to_folder, filename, model_type, activation, layers):
    if model_type == "ffnn":
        nn_model = ChassFfnnRegressor(INPUTS, layers, OUTPUTS, activation)
    elif model_type == "ernn":
        nn_model = ChassRnnRegressor(INPUTS, layers, NUM_HIDDEN, OUTPUTS, activation)
    nn_model.load_state_dict(torch.load(os.path.join(path_to_folder, filename), weights_only=True, map_location=torch.device('cpu')))
    nn_model.eval()
    train_val_test_name = f"{filename}_train_validation_test.pkl"
    train_val_test_path = os.path.join(path_to_folder, train_val_test_name)
    with open(train_val_test_path, "rb") as f:
        train_val_test = pickle.load(f)
    return nn_model, train_val_test

In [86]:
fname = "../Simulation_data.xlsx"
data = pandas.read_excel(fname, sheet_name=1)

In [87]:
def convert_symbol_to_label(symbol):
    return f"{symbol}_noise"

def convert_symbol_to_sd(symbol):
    return f"sd_{symbol}"

In [88]:
with open("../chass_testbatch.txt", "r") as f:
    testbatch_str = f.read().split()
    testbatch = set([int(el) for el in testbatch_str])
prev_val = 0
batches_lengths = []
for val in data["Experiment"]:
    if val != prev_val:
        batches_lengths.append(1)
        prev_val = val
    else:
        batches_lengths[-1] += 1
not_testbatch = set(range(len(batches_lengths))) - testbatch
not_testbatch_exp_labels = [el + 1 for el in not_testbatch]

In [89]:
maxes = dict()
mins = dict()
concentrations_labels = ['adp', 'asa', 'asp', 'aspp', 'atp', 'hs', 'hsp', 'nadp', 'nadph', 'phos', 'thr']
for label in (['time'] + concentrations_labels):
    max_val = max(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    min_val = min(data[data["Experiment"].isin(not_testbatch_exp_labels)][label])
    label_scaled = f"{label}_scaled"
    maxes[label] = max_val
    mins[label] = min_val
    data[label_scaled] = (data[label] - min_val) / (max_val - min_val)
mins_tensor = torch.Tensor([mins[label] for label in concentrations_labels])
maxes_tensor = torch.Tensor([maxes[label] for label in concentrations_labels])

In [90]:

batches = []
for i, length in enumerate(batches_lengths):
    batches.append(data.iloc[i*length : i*length + length].reset_index(drop=True))
length = batches_lengths[0]

#scale inputs
changing_vars = ['asp', 'atp']
first_of_each_batch = data[changing_vars].iloc[::length]
mins_Xs = first_of_each_batch.min()
maxes_Xs = first_of_each_batch.max()
scaled_initial_conditions = (first_of_each_batch - mins_Xs)/(maxes_Xs-mins_Xs)

In [91]:
def get_train_val_test_data(train_val_test, batches):
    trainbatch, valbatch, testbatch = train_val_test
    Xs_train = []
    ys_train = []
    Xs_val = []
    ys_val = []
    Xs_test = []
    ys_test = []
    for i, batch in enumerate(batches):
        scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
        X = pandas.concat([batch[["time_scaled"]], scaled_initials_for_this_batch], axis=1)
        y = batch[concentrations_labels]
        X = torch.Tensor(X.values)
        y = torch.Tensor(y.values)
        if i in trainbatch:
            Xs_train.append(X)
            ys_train.append(y)
        elif i in valbatch:
            Xs_val.append(X)
            ys_val.append(y)
        elif i in testbatch:
            Xs_test.append(X)
            ys_test.append(y)
    Xs_train = torch.stack(Xs_train)
    Xs_val = torch.stack(Xs_val)
    Xs_test = torch.stack(Xs_test)
    ys_train_true = torch.flatten(torch.stack(ys_train))
    ys_val_true = torch.flatten(torch.stack(ys_val))
    ys_test_true = torch.flatten(torch.stack(ys_test))
    return Xs_train, Xs_val, Xs_test, ys_train_true, ys_val_true, ys_test_true

In [92]:
Xs_ys = []
for i, batch in enumerate(batches):
    scaled_initials_for_this_batch = pandas.concat([scaled_initial_conditions.iloc[i:i+1]]*length, ignore_index=True)
    X = pandas.concat([batch[["time_scaled"]], scaled_initials_for_this_batch], axis=1)
    y = batch[[f"{label}_scaled" for label in concentrations_labels]]
    X = torch.Tensor(X.values)
    y = torch.Tensor(y.values)
    Xs_ys.append((X, y))

In [93]:
def calc_loss(nn_model, nums):
    this_batches = [batch for i, batch in enumerate(Xs_ys) if i in nums]
    X_train = torch.stack([batch[0] for batch in this_batches])
    y_train = torch.stack([batch[1] for batch in this_batches])
    
    data = TensorDataset(X_train, y_train)
    dataloader = DataLoader(data, len(data))
    sum_loss = 0
    num_loss = 0
    for X, y in dataloader:
        y_pred = nn_model(X)
        loss = loss_fn(y_pred, y)
        sum_loss += loss
        num_loss += 1
    return float(sum_loss / num_loss)

In [94]:
def make_layers_str(layers):
    if type(layers) is list:
        layers_str = "_".join([str(l) for l in layers])
    else:
        layers_str = str(layers)
    return layers_str

In [95]:
path_to_folder = "../chass_best_models/"
def open_best_model(num_nodes_layers, model_type, activation):
    layers_str = make_layers_str(num_nodes_layers)
    if model_type == "ernn":
        model_type_conv = "rnn"
    else:
        model_type_conv = model_type
    file_name = f"Chass_{model_type_conv}_{activation}_{layers_str}_best"
    with torch.no_grad():
        nn_model, train_val_test = load_model(path_to_folder, file_name, model_type, activation, num_nodes_layers)
    return nn_model, train_val_test

In [96]:
with torch.no_grad():
    for model_type in ("ffnn", "ernn"):
        for activation in ("tanh", "relu"):
            for num_nodes_layers in num_nodes[model_type]:
                nn_model, train_val_test = open_best_model(num_nodes_layers, model_type, activation)
                n_parameters = sum(p.numel() for p in nn_model.parameters() if p.requires_grad)
                print(model_type, activation, num_nodes_layers, n_parameters)
                Xs_train, Xs_val, Xs_test, ys_train_true, ys_val_true, ys_test_true = get_train_val_test_data(train_val_test, batches)
                nn_model.eval()
                y_pred_train = torch.flatten(nn_model(Xs_train) * (maxes_tensor - mins_tensor) + mins_tensor)
                y_pred_val = torch.flatten(nn_model(Xs_val) * (maxes_tensor - mins_tensor) + mins_tensor)
                y_pred_test = torch.flatten(nn_model(Xs_test) * (maxes_tensor - mins_tensor) + mins_tensor)
                R2 = sklearn.metrics.r2_score(ys_train_true, y_pred_train)
                Q2_val = sklearn.metrics.r2_score(ys_val_true, y_pred_val)
                Q2 = sklearn.metrics.r2_score(ys_test_true, y_pred_test)
                print(f"R2={R2}, Q2_val={Q2_val} Q2={Q2}")

ffnn tanh [5, 5] 116
R2=0.9633371266285482, Q2_val=0.9648168677148998 Q2=0.9602987522373729
ffnn tanh [10, 10] 271
R2=0.96734585195726, Q2_val=0.9603427622813301 Q2=0.9621447062994629
ffnn tanh [16, 16] 523
R2=0.9670964541553808, Q2_val=0.9577035657822839 Q2=0.962773684968996
ffnn tanh [16, 16, 16] 795
R2=0.9656636320550622, Q2_val=0.9651064500288997 Q2=0.958368929518578
ffnn relu [5, 5] 116
R2=0.961615347869156, Q2_val=0.963954412565827 Q2=0.9582098216025773
ffnn relu [10, 10] 271
R2=0.9671431944254553, Q2_val=0.9602854588743733 Q2=0.9621832638288579
ffnn relu [16, 16] 523
R2=0.96534698742505, Q2_val=0.9672792509122066 Q2=0.9628025656374036
ffnn relu [16, 16, 16] 795
R2=0.9652503230285481, Q2_val=0.9658948342205547 Q2=0.960777703797466
ernn tanh 10 271
R2=0.969732029762674, Q2_val=0.9366085993419972 Q2=0.9591298845568821
ernn tanh 15 476
R2=0.9704846799396492, Q2_val=0.9406978656318206 Q2=0.9624060025692328
ernn tanh 18 623
R2=0.9679178729473602, Q2_val=0.9606627922495689 Q2=0.9624970